<a href="https://colab.research.google.com/github/IrineuBovoJunior398/P-S_GRADUA-O_IA_UTFPR/blob/main/C%C3%B3pia_de_07_EXERC%C3%8DCIO___ATIVIDADE_PR%C3%81TICA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import requests
from sklearn.model_selection import cross_val_score, train_test_split, LeaveOneOut, KFold
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import LabelEncoder

# Função para carregar ARFF manualmente
def load_arff_manual(url):
    response = requests.get(url)
    content = response.text

    # Encontrar a seção de dados
    data_start = content.upper().find('@DATA')
    if data_start == -1:
        raise ValueError("Seção @DATA não encontrada")

    # Extrair metadados
    header_section = content[:data_start]
    attributes = []
    for line in header_section.split('\n'):
        if line.upper().startswith('@ATTRIBUTE'):
            # Extrair nome do atributo
            parts = line.split()
            attr_name = parts[1]
            attributes.append(attr_name)

    # Extrair apenas os dados
    data_section = content[data_start + 5:].strip()

    # Processar linhas
    lines = [line.strip() for line in data_section.split('\n') if line.strip() and not line.strip().startswith('%')]

    # Converter para lista de valores
    data = []
    for line in lines:
        values = [v.strip() for v in line.split(',')]
        data.append(values)

    return np.array(data), attributes

# Carregar dataset
url = 'https://www.openml.org/data/download/3647/dataset_2196_cloud.arff'
try:
    data, attributes = load_arff_manual(url)
    print(f"Dataset carregado com sucesso!")
    print(f"Atributos: {attributes}")
    print(f"Shape dos dados: {data.shape}")

    # Criar DataFrame
    df = pd.DataFrame(data, columns=attributes)

    # Limpar nomes de colunas (remover aspas)
    df.columns = [col.strip("'") for col in df.columns]

    # Converter para tipos apropriados
    for col in df.columns[:-1]:  # Todas exceto a última (target)
        try:
            df[col] = pd.to_numeric(df[col])
        except:
            # Se não for numérico, fazer encoding
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col])

    # Target pode ser numérico ou categórico
    try:
        df[df.columns[-1]] = pd.to_numeric(df[df.columns[-1]])
    except:
        le = LabelEncoder()
        df[df.columns[-1]] = le.fit_transform(df[df.columns[-1]])

    # Separar features e target
    X = df.iloc[:, :-1].astype(float)
    y = df.iloc[:, -1].astype(float)

    print(f"\nFeatures: {list(X.columns)}")
    print(f"Target: {df.columns[-1]}")
    print(f"Dimensões X: {X.shape}, y: {y.shape}")
    print(f"\nPrimeiras 5 linhas:")
    print(df.head())

    print("\n" + "="*70)
    print("ANÁLISE COMPLETA DO DATASET CLOUD")
    print("="*70)

    print("\n" + "="*70)
    print("ATIVIDADE 1: 10 EXECUÇÕES DE 5-FOLDS CV COM DIFERENTES RANDOM_STATE")
    print("="*70)

    r2_scores = []
    for i in range(10):
        kfold = KFold(n_splits=5, shuffle=True, random_state=i)
        model = LinearRegression()
        scores = cross_val_score(model, X, y, cv=kfold, scoring='r2')
        r2_scores.append(np.mean(scores))
        print(f"Execução {i+1} (random_state={i}): R² médio = {np.mean(scores):.4f}")

    avg_r2 = np.mean(r2_scores)
    std_r2 = np.std(r2_scores)
    print(f"\n✓ R² MÉDIO GERAL (10 execuções): {avg_r2:.4f}")
    print(f"  Desvio padrão: {std_r2:.4f}")
    print(f"  Mínimo: {np.min(r2_scores):.4f}")
    print(f"  Máximo: {np.max(r2_scores):.4f}")

    print("\n" + "="*70)
    print("ATIVIDADE 2: MODELO LASSO COM ALPHA=1 - COEFICIENTES ZERADOS")
    print("="*70)

    lasso = Lasso(alpha=1, max_iter=10000)
    lasso.fit(X, y)
    coef_lasso = lasso.coef_

    print("\nCoeficientes do Lasso (alpha=1):")
    for col, coef in zip(X.columns, coef_lasso):
        status = "ZERADO" if abs(coef) < 1e-6 else "NÃO-ZERADO"
        print(f"  {col}: {coef:.6f} [{status}]")

    zero_coefs = [col for col, coef in zip(X.columns, coef_lasso) if abs(coef) < 1e-6]
    num_zero = len(zero_coefs)
    print(f"\n✓ QUANTIDADE DE COEFICIENTES ZERADOS: {num_zero}")
    if zero_coefs:
        print(f"  Atributos com coeficiente zerado: {zero_coefs}")
    else:
        print(f"  Nenhum atributo teve seu coeficiente zerado com alpha=1")

    print("\n" + "="*70)
    print("ATIVIDADE 3: REGRESSÃO LINEAR - COEFICIENTES DE CADA ATRIBUTO")
    print("="*70)

    lin_reg = LinearRegression()
    lin_reg.fit(X, y)
    coef_lin = lin_reg.coef_

    print("\nCoeficientes da Regressão Linear:")
    for col, coef in zip(X.columns, coef_lin):
        print(f"  {col}: {coef:.6f}")

    print(f"\nIntercept (β₀): {lin_reg.intercept_:.6f}")

    print("\n" + "="*70)
    print("ATIVIDADE 4: COMPARAÇÃO 80/20 - LINEAR, LASSO E RIDGE (RMSE)")
    print("="*70)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    print(f"\nTamanho do conjunto de treinamento: {X_train.shape[0]}")
    print(f"Tamanho do conjunto de teste: {X_test.shape[0]}")

    models = {
        'Regressão Linear': LinearRegression(),
        'Lasso (alpha=1)': Lasso(alpha=1, max_iter=10000),
        'Ridge (alpha=1)': Ridge(alpha=1)
    }

    rmse_results = {}
    print("\nResultados RMSE no conjunto de teste:")
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        rmse_results[name] = rmse
        print(f"  {name}: {rmse:.4f}")

    best_model = min(rmse_results, key=rmse_results.get)
    print(f"\n✓ MELHOR MODELO: {best_model} (RMSE = {rmse_results[best_model]:.4f})")

    print("\n" + "="*70)
    print("ATIVIDADE 5: LEAVE-ONE-OUT CV - REGRESSÃO LINEAR (MAE)")
    print("="*70)

    loo = LeaveOneOut()
    model = LinearRegression()
    mae_scores = cross_val_score(model, X, y, cv=loo, scoring='neg_mean_absolute_error')
    mae_mean = -np.mean(mae_scores)
    mae_std = np.std(-mae_scores)

    print(f"\n✓ MAE MÉDIO (Leave-One-Out): {mae_mean:.4f}")
    print(f"  Desvio padrão: {mae_std:.4f}")
    print(f"  Total de iterações (LOO): {len(mae_scores)}")
    print(f"  Mínimo MAE: {np.min(-mae_scores):.4f}")
    print(f"  Máximo MAE: {np.max(-mae_scores):.4f}")

    print("\n" + "="*70)
    print("RESUMO FINAL")
    print("="*70)
    print(f"1. R² médio (10x 5-folds CV): {avg_r2:.4f} (±{std_r2:.4f})")
    print(f"2. Coeficientes zerados (Lasso, alpha=1): {num_zero}")
    print(f"3. Coeficientes (Regressão Linear): Veja acima")
    print(f"4. Melhor modelo (80/20): {best_model} (RMSE={rmse_results[best_model]:.4f})")
    print(f"5. MAE (Leave-One-Out): {mae_mean:.4f}")
    print("="*70)

except Exception as e:
    print(f"Erro: {e}")
    import traceback
    traceback.print_exc()

Dataset carregado com sucesso!
Atributos: ["'period'", "'seeded'", "'season'", "'NC'", "'SC'", "'NWC'", "'TE'"]
Shape dos dados: (108, 7)

Features: ['period', 'seeded', 'season', 'NC', 'SC', 'NWC']
Target: TE
Dimensões X: (108, 6), y: (108,)

Primeiras 5 linhas:
   period  seeded  season    NC    SC   NWC    TE
0       1       0       0  1.65  1.80  3.33  1.69
1       2       1       0  1.09  0.79  1.59  0.74
2       3       0       3  2.39  0.36  2.06  0.81
3       4       1       3  2.96  1.27  4.05  1.44
4       5       0       3  4.16  2.16  6.00  2.48

ANÁLISE COMPLETA DO DATASET CLOUD

ATIVIDADE 1: 10 EXECUÇÕES DE 5-FOLDS CV COM DIFERENTES RANDOM_STATE
Execução 1 (random_state=0): R² médio = 0.8336
Execução 2 (random_state=1): R² médio = 0.8300
Execução 3 (random_state=2): R² médio = 0.8138
Execução 4 (random_state=3): R² médio = 0.8252
Execução 5 (random_state=4): R² médio = 0.8667
Execução 6 (random_state=5): R² médio = 0.8247
Execução 7 (random_state=6): R² médio = 0.8406
Exe